# Graph U-Net for Landslide Segmentation
**Pretrain on Hokkaido, then fine-tune on Bijie**

This notebook is a custom superpixel-based experiment inspired by
[Graph U-Nets](https://proceedings.mlr.press/v97/gao19a.html). It is separate
from the pixel-grid implementation in this repository and from the AMMG-UNet
architecture in the landslide transfer-learning paper.

Repository reference: [HongyangGao/Graph-U-Nets](https://github.com/HongyangGao/Graph-U-Nets)

---
**Notebook flow**
```
Step 1   Install and import dependencies
Step 2   Configure explicit dataset paths
Step 3   Inspect raw image/mask pairs
Step 4   Build SLIC superpixel graphs
Step 5   Create deterministic train/validation/test splits
Step 6   Define the custom Graph U-Net
Step 7   Define loss, metrics, and training loop
Step 8   Pretrain on Hokkaido
Step 9   Fine-tune on Bijie under four conditions
Step 10  Report held-out test results
Step 11  Visualize predictions
```

> The default 50/15 epoch schedule is a practical demonstration setting.
> Increase it only after validating the dataset, masks, and split manifests.


## Step 1 · Install

In [ ]:
%pip install rasterio scikit-image -q
print('Done')

In [ ]:
# Optional: mount Google Drive only when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted (Colab).')
except Exception:
    print('Colab not detected; using local filesystem.')

In [ ]:
import os, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
import rasterio
from rasterio.enums import Resampling
from skimage.segmentation import slic, mark_boundaries

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
print(f'Device: {DEVICE}')


## Step 2 - Paths and configuration

Set `HOKKAIDO_ROOT` and `BIJIE_ROOT` to the two dataset directories before
running the notebook. If an environment variable is omitted, the notebook uses
`dataset/hokkaido` or `dataset/bijie` relative to the current directory.

Each root must contain an image directory (`img`, `image`, or `images`) and a
mask directory (`mask`, `masks`, `label`, or `labels`). The dataset files are
intentionally excluded from Git.


In [ ]:
import os

In [ ]:
from pathlib import Path
import os

CWD = Path.cwd()

def resolve_dataset_root(env_var, relative_default):
    configured = os.getenv(env_var)
    candidate = Path(configured).expanduser() if configured else CWD / relative_default
    candidate = candidate.resolve()
    if not candidate.is_dir():
        raise FileNotFoundError(
            f'Dataset directory not found: {candidate}. '
            f'Set {env_var} to the downloaded dataset root.'
        )
    return str(candidate)

HOKKAIDO = resolve_dataset_root('HOKKAIDO_ROOT', 'dataset/hokkaido')
BIJIE = resolve_dataset_root('BIJIE_ROOT', 'dataset/bijie')

CKPT_DIR = Path(os.getenv('CHECKPOINT_DIR', './ckpts')).expanduser()
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Image and graph
IMG_SIZE = 512
N_SEGS = 500  # reduce to 250 for faster debugging

# Model
HIDDEN = 64
DEPTH = 2
RATIO = 0.5
DROP = 0.1

# Practical demonstration schedule
PRE_EP = 50; PRE_LR = 1e-2; PRE_WD = 1e-3
FT_EP = 15; FT_LR = 1e-3; FT_WD = 1e-3

print('Config set.')
print('HOKKAIDO =', HOKKAIDO)
print('BIJIE    =', BIJIE)


## Step 3 · Look at the Raw Data

Expected folder layout for **each** dataset:
```
<root>/
  img/   ← .tif satellite images  (512×512 RGB)
  mask/  ← .tif binary masks      (1 = landslide, 0 = background)
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling

EXTS = {'.tif','.tiff','.png','.jpg','.jpeg'}
IMG_DIR_NAMES = ('img', 'image', 'images')
MASK_DIR_NAMES = ('mask', 'masks', 'label', 'labels')


def find_existing_subdir(root, names):
    root = Path(root)
    for name in names:
        direct = root / name
        if direct.is_dir():
            return direct
    for name in names:
        for candidate in root.glob(f'**/{name}'):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError(f'Could not find any of {names} under {root}')


def get_pairs(root):
    root = Path(root)
    img_dir = find_existing_subdir(root, IMG_DIR_NAMES)
    mask_dir = find_existing_subdir(root, MASK_DIR_NAMES)
    pairs = []
    for f in sorted(img_dir.iterdir()):
        if f.suffix.lower() in EXTS:
            m = mask_dir / f.name
            if not m.exists():
                m = mask_dir / (f.stem + '.tif')
            if not m.exists():
                m = mask_dir / (f.stem + '.png')
            if not m.exists():
                m = mask_dir / (f.stem + '.jpg')
            if m.exists():
                pairs.append((str(f), str(m)))
    print(f'  {root.name}: {len(pairs)} pairs')
    print(f'    images: {img_dir}')
    print(f'    masks : {mask_dir}')
    return pairs


def read_img(p):
    with rasterio.open(p) as s:
        d = s.read(out_shape=(s.count,IMG_SIZE,IMG_SIZE),
                   resampling=Resampling.bilinear).astype(np.float32)
    d = d[:3] if d.shape[0]>=3 else np.repeat(d[:1],3,0)
    for i in range(3):
        lo,hi=d[i].min(),d[i].max(); d[i]=(d[i]-lo)/(hi-lo+1e-7)
    return d.transpose(1,2,0)          # (H,W,3)


def read_mask(p):
    with rasterio.open(p) as s:
        m = s.read(1,out_shape=(IMG_SIZE,IMG_SIZE),
                   resampling=Resampling.nearest)
    return (m>0).astype(np.uint8)

print('Loading pairs...')
hok_pairs   = get_pairs(HOKKAIDO)
bijie_pairs = get_pairs(BIJIE)

# Show 2 samples from each dataset
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Raw Dataset Samples', fontsize=14, fontweight='bold')

for row, (pairs, name) in enumerate([(hok_pairs,'Hokkaido'), (bijie_pairs,'Bijie')]):
    for col, idx in enumerate([0, 1]):
        img  = read_img(pairs[idx][0])
        mask = read_mask(pairs[idx][1])
        pct  = mask.mean()*100
        axes[row][col*2  ].imshow(img)
        axes[row][col*2  ].set_title(f'{name} [{idx}] — Image'); axes[row][col*2].axis('off')
        axes[row][col*2+1].imshow(mask, cmap='gray')
        axes[row][col*2+1].set_title(f'Mask  ({pct:.1f}% landslide)')
        axes[row][col*2+1].axis('off')

plt.tight_layout(); plt.show()

## Step 4 · SLIC Superpixels → Graph

Each image becomes a graph:

| Graph element | Meaning |
|---|---|
| **Node** | One SLIC superpixel (group of similar pixels) |
| **Node feature** | Mean R, G, B of that superpixel |
| **Edge** | Two superpixels share a boundary |
| **Node label** | 1 if ≥50% of its pixels are landslide, else 0 |

In [ ]:
# ── Visualise the pipeline on one image ─────────────────────────────────────
def show_pipeline(img_path, mask_path, title):
    img  = read_img(img_path)
    mask = read_mask(mask_path)
    seg  = slic(img, n_segments=N_SEGS, compactness=10,
                start_label=0, sigma=1)

    # remap segment ids to 0..N-1
    uq = np.unique(seg)
    seg = np.vectorize({v:i for i,v in enumerate(uq)}.get)(seg)
    N   = len(uq)

    # node label = majority vote
    node_lbl = np.array([1 if mask[seg==i].mean()>=0.5 else 0
                         for i in range(N)])
    lbl_map  = node_lbl[seg]   # (H,W) for display

    fig, ax = plt.subplots(1,4,figsize=(20,4.5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    ax[0].imshow(img)
    ax[0].set_title('① Original image'); ax[0].axis('off')

    ax[1].imshow(mark_boundaries(img, seg, color=(1,.3,0)))
    ax[1].set_title(f'② SLIC superpixels  (N={N})'); ax[1].axis('off')

    ax[2].imshow(mask, cmap='gray')
    ax[2].set_title('③ Ground-truth mask'); ax[2].axis('off')

    cmap2 = plt.matplotlib.colors.ListedColormap(['#222222','#e03030'])
    ax[3].imshow(lbl_map, cmap=cmap2, vmin=0, vmax=1)
    ax[3].imshow(mark_boundaries(img,seg,color=(1,1,1)), alpha=0.2)
    p = [mpatches.Patch(color='#e03030',label=f'Landslide ({node_lbl.sum()})'),
         mpatches.Patch(color='#222222',label=f'Background ({(node_lbl==0).sum()})')]
    ax[3].legend(handles=p, loc='upper right', fontsize=8)
    ax[3].set_title('④ Node labels'); ax[3].axis('off')

    plt.tight_layout(); plt.show()
    print(f'   Nodes: {N}  |  Landslide nodes: {node_lbl.mean()*100:.1f}%')

show_pipeline(*hok_pairs[0],   'Hokkaido sample')
show_pipeline(*bijie_pairs[0], 'Bijie sample')

## Step 5 · Build Graphs + **Strict Split**

 
> If the same image appears in both train and val the model memorises it.
> Here we:
> 1. Shuffle indices with a fixed seed
> 2. Slice into **non-overlapping** train / val / test
> 3. **Assert** zero overlap before any training begins

In [ ]:
def build_graph(img_path, mask_path):
    """Return dict: feat (N,3), adj_norm (N,N), labels (N,1), seg_map (H,W)."""
    img  = read_img(img_path)
    mask = read_mask(mask_path)
    seg  = slic(img, n_segments=N_SEGS, compactness=10,
                start_label=0, sigma=1)
    uq   = np.unique(seg)
    seg  = np.vectorize({v:i for i,v in enumerate(uq)}.get)(seg)
    N    = len(uq)

    feat = np.stack([img[seg==i].mean(0) if (seg==i).any() else np.zeros(3) for i in range(N)]).astype(np.float32)

    A = np.zeros((N,N), np.float32)
    for dy,dx in [(0,1),(1,0)]:
            if dy == 0 and dx == 1:
                s1 = seg[:, :-1]
                s2 = seg[:, 1:]
            else:
                s1 = seg[:-1, :]
                s2 = seg[1:, :]
            bd = s1 != s2
            for a,b in zip(s1[bd], s2[bd]): A[a,b] = A[b,a] = 1

    At = torch.from_numpy(A)+torch.eye(N)
    d  = At.sum(1).pow(-0.5); d[torch.isinf(d)]=0
    D  = torch.diag(d)
    An = (D@At@D).float()

    lbl = np.array([1 if mask[seg==i].mean()>=0.5 else 0
                    for i in range(N)], np.float32)

    return dict(feat=torch.from_numpy(feat),
                adj_norm=An,
                labels=torch.from_numpy(lbl).unsqueeze(-1),
                seg_map=seg, img_path=img_path, mask_path=mask_path)

# Cache built graphs to avoid recomputing SLIC for every epoch (speeds training).
# Keeps a small in-memory cache keyed by (img_path, mask_path).
try:
    from functools import lru_cache
    _orig_build_graph = build_graph
    @lru_cache(maxsize=4096)
    def _build_graph_cache(img_path, mask_path):
        return _orig_build_graph(img_path, mask_path)
    build_graph = _build_graph_cache
    print('build_graph: caching enabled')
except Exception as e:
    print('build_graph: caching not enabled:', e)


# ── Strict split ─────────────────────────────────────────────────────────────
def strict_split(n, tr=0.6, va=0.2, seed=SEED):
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    a, b = int(n*tr), int(n*(tr+va))
    train, val, test = idx[:a], idx[a:b], idx[b:]

    # ── ZERO OVERLAP ASSERTION ────────────────────────────────────────────
    assert not set(train)&set(val),  'BUG: train/val overlap!'
    assert not set(train)&set(test), 'BUG: train/test overlap!'
    assert not set(val)  &set(test), 'BUG: val/test overlap!'
    print(f'  train={len(train)}  val={len(val)}  test={len(test)}  '
          f'total={len(train)+len(val)+len(test)} ✓ no overlap')
    return train, val, test


print('Hokkaido split:')
HOK_TR, HOK_VA, HOK_TE = strict_split(len(hok_pairs))

print('Bijie split:')
BIJ_TR60, BIJ_VA, BIJ_TE = strict_split(len(bijie_pairs))
# 20% condition = first third of the 60% training set
BIJ_TR20 = BIJ_TR60[:max(1, len(BIJ_TR60)//3)]
print(f'  20%-condition train size: {len(BIJ_TR20)}')

# Final sanity check — 20% train must not touch val/test
assert not set(BIJ_TR20)&set(BIJ_VA), 'BUG'
assert not set(BIJ_TR20)&set(BIJ_TE), 'BUG'
print('All splits verified ✓')

## Step 6 - Custom Graph U-Net model

This model is inspired by
[Gao and Ji (2019)](https://proceedings.mlr.press/v97/gao19a.html), but it is a
custom adaptation: nodes are SLIC superpixels, node features are mean RGB,
decoder skips are concatenated, and the graph operations differ from the
authors' citation-network implementation. Results should therefore be described
as a Graph U-Net experiment, not as an exact reproduction of the reference repo.

```
x (N, 3)  --> embed GCN
              |-- encoder GCN + gPool  --> N/2 nodes -- skip 1
              |   encoder GCN + gPool  --> N/4 nodes -- skip 2
              |       bottleneck GCN
              |   gUnpool + concat(skip 2) + decoder GCN
              +-- gUnpool + concat(skip 1) + decoder GCN
                  output Linear --> one binary logit per node
```

**gPool** scores nodes with a learnable vector, keeps the top-k nodes, and gates
their features. **gUnpool** restores retained nodes to their earlier positions.


In [ ]:
class GCN(nn.Module):
    def __init__(self, in_c, out_c, drop=0.0):
        super().__init__()
        self.W  = nn.Linear(in_c, out_c, bias=False)
        self.bn = nn.BatchNorm1d(out_c)
        self.dr = nn.Dropout(drop)
    def forward(self, x, A):
        h = self.W(A @ x)
        h = self.bn(h) if h.size(0)>1 else h
        return self.dr(F.relu(h))


class gPool(nn.Module):
    def __init__(self, dim, ratio=0.5):
        super().__init__()
        self.p     = nn.Parameter(torch.randn(dim))
        self.ratio = ratio
    def forward(self, x, A):
        n   = x.size(0)
        k   = max(1, int(self.ratio * n))
        p   = self.p / (self.p.norm() + 1e-8)
        y   = (x @ p)                          # (N,)  projection score
        idx = y.topk(k)[1].sort()[0]           # top-k indices
        xp  = x[idx] * torch.sigmoid(y[idx]).unsqueeze(-1)
        Ap  = A[idx][:, idx]
        # renormalise sub-adjacency
        Ap  = Ap + torch.eye(k, device=Ap.device)
        d   = Ap.sum(1).pow(-0.5); d[torch.isinf(d)]=0
        Ap  = torch.diag(d) @ Ap @ torch.diag(d)
        return xp, Ap, idx, n


class gUnpool(nn.Module):
    def forward(self, xp, idx, n_orig, A_orig):
        x = torch.zeros(n_orig, xp.size(-1),
                        device=xp.device, dtype=xp.dtype)
        x[idx] = xp
        return x, A_orig


class GraphUNet(nn.Module):
    def __init__(self, in_c=3, hid=64, depth=2, ratio=0.5, drop=0.1):
        super().__init__()
        self.depth   = depth
        self.embed   = GCN(in_c, hid, drop)
        self.enc     = nn.ModuleList([GCN(hid,hid,drop) for _ in range(depth)])
        self.pools   = nn.ModuleList([gPool(hid,ratio)  for _ in range(depth)])
        self.btn     = GCN(hid,hid,drop)
        self.dec     = nn.ModuleList([GCN(hid*2,hid,drop) for _ in range(depth)])
        self.unpools = nn.ModuleList([gUnpool()            for _ in range(depth)])
        self.head    = nn.Linear(hid, 1)

    def forward(self, x, A):
        x = self.embed(x, A)
        skips, As, idxs, ns = [], [], [], []
        for gcn, pool in zip(self.enc, self.pools):
            x = gcn(x, A)
            skips.append(x); As.append(A)
            x, A, idx, n = pool(x, A)
            idxs.append(idx); ns.append(n)
        x = self.btn(x, A)
        for i,(gcn,unpool) in enumerate(zip(self.dec,self.unpools)):
            ri = self.depth-1-i
            x, A = unpool(x, idxs[ri], ns[ri], As[ri])
            x = torch.cat([x, skips[ri]], dim=-1)
            x = gcn(x, A)
        return self.head(x)   # (N,1) raw logits


# smoke test
_m  = GraphUNet(3,HIDDEN,DEPTH,RATIO,DROP).to(DEVICE)
_x  = torch.randn(500,3).to(DEVICE)
_A  = torch.eye(500).to(DEVICE)
with torch.no_grad(): _o = _m(_x,_A)
print(f'Model OK  →  output {tuple(_o.shape)}')
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m,_x,_A,_o

## Step 7 · Loss, Metrics, Training Loop

In [ ]:
# ── Loss: Dice + 0.5 × BCE ──────────────────────────────────────────────────
def seg_loss(logits, labels):
    p = torch.sigmoid(logits).view(-1)
    g = labels.view(-1)
    dice = 1 - (2*(p*g).sum()+1e-6)/(p.sum()+g.sum()+1e-6)
    bce  = F.binary_cross_entropy_with_logits(logits.view(-1), g)
    return dice + 0.5*bce


# ── Metrics ───────────────────────────────────────────────────────────────────
def metrics(logits, labels, thr=0.5):
    pred = (torch.sigmoid(logits)>=thr).long().view(-1).cpu()
    y    = labels.long().view(-1).cpu()
    tp=int((pred*y).sum()); fp=int((pred*(1-y)).sum())
    fn=int(((1-pred)*y).sum()); tn=int(((1-pred)*(1-y)).sum())
    e=1e-7
    R=tp/(tp+fn+e)*100; Sp=tn/(tn+fp+e)*100
    Pr=tp/(tp+fp+e)*100; F1=2*Pr*R/(Pr+R+e)
    return dict(F1=round(F1,2),Recall=round(R,2),
                Specificity=round(Sp,2),Precision=round(Pr,2))


# ── One epoch ─────────────────────────────────────────────────────────────────
def one_epoch(model, pairs, indices, opt=None):
    """Train if opt given, else eval. One graph per step with visible progress."""
    model.train() if opt else model.eval()
    tot, all_l, all_p = 0.0, [], []
    idx_list = list(indices)
    if opt:
        random.shuffle(idx_list)

    phase = 'train-graphs' if opt else 'val-graphs'
    loop = tqdm(idx_list, desc=phase, leave=False, ncols=100)

    ctx = torch.enable_grad() if opt else torch.no_grad()
    with ctx:
        for i in loop:
            g  = build_graph(*pairs[i])
            x  = g['feat'].to(DEVICE)
            A  = g['adj_norm'].to(DEVICE)
            lb = g['labels'].to(DEVICE)
            lo = model(x, A)
            ls = seg_loss(lo, lb)
            if opt:
                opt.zero_grad(set_to_none=True)
                ls.backward()
                nn.utils.clip_grad_norm_(model.parameters(),5)
                opt.step()
            tot += ls.item()
            all_l.append(lo.detach().cpu())
            all_p.append(lb.cpu())
            loop.set_postfix(loss=f"{ls.item():.4f}")

    m = metrics(torch.cat(all_l), torch.cat(all_p))
    return tot/max(1, len(idx_list)), m


# ── Full training ─────────────────────────────────────────────────────────────
def train(model, pairs, tr_idx, va_idx, epochs, lr, wd, label):
    """Returns best model (by val F1) and history list."""
    opt   = torch.optim.SGD(model.parameters(),lr=lr,momentum=0.9,weight_decay=wd)
    sched = LambdaLR(opt, lambda e:(1-min(e,epochs)/epochs)**3)
    best_f1, best_st, hist = 0.0, deepcopy(model.state_dict()), []

    for ep in tqdm(range(epochs), desc=label, ncols=80):
        tl,_  = one_epoch(model, pairs, tr_idx, opt)
        vl,vm = one_epoch(model, pairs, va_idx)
        sched.step()
        hist.append(dict(ep=ep+1,tl=tl,vl=vl,**vm))
        if vm['F1']>best_f1:
            best_f1=vm['F1']; best_st=deepcopy(model.state_dict())

    model.load_state_dict(best_st)
    return hist, best_f1


def plot_hist(hist, title):
    ep=[h['ep'] for h in hist]
    fig,ax=plt.subplots(1,2,figsize=(11,3.5))
    fig.suptitle(title,fontweight='bold')
    ax[0].plot(ep,[h['tl'] for h in hist],label='train')
    ax[0].plot(ep,[h['vl'] for h in hist],label='val')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend()
    ax[1].plot(ep,[h['F1']        for h in hist],label='F1')
    ax[1].plot(ep,[h['Recall']    for h in hist],label='Recall')
    ax[1].plot(ep,[h['Precision'] for h in hist],label='Precision')
    ax[1].set_xlabel('epoch'); ax[1].set_ylabel('%'); ax[1].legend()
    plt.tight_layout(); plt.show()

print('Ready.')

## Step 8 - Phase 1: pretrain on Hokkaido

The default is 50 epochs with SGD (`lr=0.01`) and cubic polynomial learning-rate
decay. The best validation checkpoint is loaded for transfer-learning runs in
Step 9.


In [ ]:
pretrain_model = GraphUNet(3,HIDDEN,DEPTH,RATIO,DROP).to(DEVICE)

pre_hist, pre_best = train(
    pretrain_model, hok_pairs,
    HOK_TR, HOK_VA,
    PRE_EP, PRE_LR, PRE_WD,
    'Pretrain — Hokkaido'
)

torch.save(pretrain_model.state_dict(), f'{CKPT_DIR}/pretrained.pth')
print(f'\nBest val F1 = {pre_best:.2f}%')
plot_hist(pre_hist, 'Pretrain on Hokkaido')

## Step 9 · Phase 2 — Fine-Tune on Bijie (4 Conditions)

| | Train set | TL | Description |
|---|---|---|---|
| A | 20% Bijie | ✗ | Scratch, limited data |
| B | 20% Bijie | ✓ | Pretrained on Hokkaido |
| C | 60% Bijie | ✗ | Scratch, more data |
| D | 60% Bijie | ✓ | Pretrained on Hokkaido |

> Test set is **never touched during training or model selection**.

In [ ]:
results = []

for ratio, tr_idx in [('20%', BIJ_TR20), ('60%', BIJ_TR60)]:
    for use_tl in [False, True]:
        label = f"{ratio} {'+ TL' if use_tl else 'scratch'}"
        print(f'\n── {label}  (train={len(tr_idx)}) ──')

        m = GraphUNet(3,HIDDEN,DEPTH,RATIO,DROP).to(DEVICE)
        if use_tl:
            m.load_state_dict(torch.load(f'{CKPT_DIR}/pretrained.pth',
                                          map_location=DEVICE))
            print('  loaded Hokkaido weights')

        hist, bv = train(m, bijie_pairs, tr_idx, BIJ_VA,
                         FT_EP, FT_LR, FT_WD, label)

        # ── evaluate on HELD-OUT test (never seen during training) ────────
        _, test_m = one_epoch(m, bijie_pairs, BIJ_TE)

        results.append(dict(Condition=label, TL='Yes' if use_tl else 'No',
                            **test_m))
        torch.save(m.state_dict(),
                   f"{CKPT_DIR}/ft_{ratio}_{'TL' if use_tl else 'DL'}.pth")

        print(f'  TEST → F1={test_m["F1"]}%  '
              f'Recall={test_m["Recall"]}%  '
              f'Spec={test_m["Specificity"]}%  '
              f'Prec={test_m["Precision"]}%')
        plot_hist(hist, label)

print('\nAll 4 conditions complete.')

## Step 10 · Results

In [ ]:
print(f'\n{"="*62}')
print(f'  Graph U-Net  —  Bijie TEST SET')
print(f'{"="*62}')
print(f'  {"Condition":<20} {"F1":>6} {"Recall":>8} {"Spec":>8} {"Prec":>8}')
print(f'  {"-"*56}')
for r in results:
    print(f"  {r['Condition']:<20} {r['F1']:>5.2f}%"
          f" {r['Recall']:>7.2f}% {r['Specificity']:>7.2f}%"
          f" {r['Precision']:>7.2f}%")
print(f'{"="*62}')
# TL delta
for rat in ['20%','60%']:
    dl = next(r['F1'] for r in results if rat in r['Condition'] and r['TL']=='No')
    tl = next(r['F1'] for r in results if rat in r['Condition'] and r['TL']=='Yes')
    print(f'  TL improvement @ {rat}: {tl-dl:+.2f}%')

## Step 11 · Visualise Predictions on Test Images

In [ ]:
best_m = GraphUNet(3,HIDDEN,DEPTH,RATIO,DROP).to(DEVICE)
best_m.load_state_dict(torch.load(f'{CKPT_DIR}/ft_60%_TL.pth',map_location=DEVICE))
best_m.eval()

for idx in BIJ_TE[:3]:
    g       = build_graph(*bijie_pairs[idx])
    x       = g['feat'].to(DEVICE)
    A       = g['adj_norm'].to(DEVICE)
    seg_map = g['seg_map']
    img     = read_img(g['img_path'])
    gt      = read_mask(g['mask_path'])

    with torch.no_grad():
        lo = best_m(x, A)
    probs = torch.sigmoid(lo).squeeze().cpu().numpy()
    preds = (probs>=0.5).astype(np.uint8)

    # map node predictions back to pixels
    prob_px = probs[seg_map]
    pred_px = preds[seg_map]

    # colour overlay: TP=green FP=blue FN=red
    ov = np.zeros((*gt.shape,3), np.uint8)
    ov[(pred_px==1)&(gt==1)]=[0,  200,0  ]
    ov[(pred_px==1)&(gt==0)]=[0,  0,  200]
    ov[(pred_px==0)&(gt==1)]=[200,0,  0  ]

    fig,ax=plt.subplots(1,4,figsize=(19,4.5))
    fig.suptitle(f'Test image index {idx}  (60% + TL)',fontweight='bold')
    ax[0].imshow(img);                     ax[0].set_title('Image');       ax[0].axis('off')
    ax[1].imshow(gt,cmap='gray');           ax[1].set_title('Ground truth');ax[1].axis('off')
    im=ax[2].imshow(prob_px,cmap='jet',vmin=0,vmax=1)
    plt.colorbar(im,ax=ax[2],fraction=0.046);ax[2].set_title('Probability');ax[2].axis('off')
    ax[3].imshow(img); ax[3].imshow(ov,alpha=0.5)
    legend_handles = [
        mpatches.Patch(color=c, label=l)
        for c, l in [([0,.78,0],'TP'),([0,0,.78],'FP'),([.78,0,0],'FN')]
    ]
    ax[3].legend(handles=legend_handles,loc='upper right',fontsize=9)
    ax[3].set_title('TP / FP / FN'); ax[3].axis('off')
    plt.tight_layout(); plt.show()